In [ ]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 97.9 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

st.title("Fake Job Recruitment Detection (with Deep Learning)")

@st.cache_data
def load_data():
    df = pd.read_csv("fake_job_postings.csv")
    df = df.fillna("")
    return df

df = load_data()
X = df["description"].values
y = df["fraudulent"].values

max_words = 10000
max_len = 300

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

X_sequences = tokenizer.texts_to_sequences(X)
X_padded = pad_sequences(X_sequences, maxlen=max_len, padding="post", truncating="post")

X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y, test_size=0.2, random_state=42
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

@st.cache_resource
def train_ann_model(X_tr, y_tr, X_ts, y_ts):
    model = Sequential([
        Embedding(max_words, 16, input_length=max_len),
        GlobalAveragePooling1D(),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X_tr, y_tr, epochs=5, batch_size=32, class_weight=class_weight_dict, verbose=0)
    return model

model = train_ann_model(X_train, y_train, X_test, y_test)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
st.write("ANN Model Accuracy:", round(accuracy * 100, 2), "%")

st.subheader("Verify a Job Posting")
job_text = st.text_area("Paste Job Description Here")

if st.button("Check Job"):
    if job_text.strip() == "":
        st.warning("Please paste some text first!")
    else:
        input_seq = tokenizer.texts_to_sequences([job_text])
        input_padded = pad_sequences(input_seq, maxlen=max_len, padding="post", truncating="post")
        prediction = model.predict(input_padded)[0][0]

        if prediction > 0.5:
            st.error(f"⚠️ Fake Job Posting! (Confidence: {round(prediction * 100, 2)}%)")
        else:
            st.success(f"✅ Genuine Job Posting. (Confidence: {round((1 - prediction) * 100, 2)}%)")

st.subheader("Career Guidance Chatbot")
question = st.text_input("Ask Career Question")

if question:
    q = question.lower()
    if "python" in q:
        st.write("Learn Python, Pandas, NumPy and Machine Learning.")
    elif "data science" in q:
        st.write("Focus on Statistics, Python and Machine Learning.")
    elif "cyber security" in q:
        st.write("Learn Networking, Linux and Ethical Hacking.")
    else:
        st.write("Please ask about Python, Data Science or Cyber Security.")

Writing app.py


In [ ]:
!wget -qO- ipv4.icanhazip.com

34.56.166.93


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501



⠙⠹⠸⠼⠴⠦Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-06-27 05:52:41.404 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.56.166.93:8501

y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋your url is: https://huge-moons-ring.loca.lt
2026-06-27 05:55:25.443190: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-27 05:55:25.490669: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  w